# 

In [2]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from dataset_ood_download import get_data_list, system_prompt
import selfies as sf

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train_path = '/data/data/BioT5_bace/BioT5_bace_train.csv'
test_path = '/data/data/BioT5_bace/BioT5_bace_test.csv'

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

In [9]:
train_data.columns

Index(['iupacname', 'SELFIES', 'label'], dtype='object')

In [12]:

train_selfies = train_data['SELFIES'].tolist()
train_labels = train_data['label'].tolist()
train_smiles = [sf.decoder(i) for i in train_selfies]
train_mols = [Chem.MolFromSmiles(i) for i in train_smiles]


test_selfies = test_data['SELFIES'].tolist()
test_labels = test_data['label'].tolist()
test_smiles = [sf.decoder(i) for i in test_selfies]
test_mols = [Chem.MolFromSmiles(i) for i in test_smiles]

In [15]:
task = 'bace'
list_train_data = get_data_list(
    list_mol=train_mols,
    list_label=train_labels,
    task='bace',
    instruction_templates=instructions_smol.bace,
)

list_test_data = get_data_list(
    list_mol=test_mols,
    list_label=test_labels,
    task='bace',
    instruction_templates=instructions_smol.bace,
)

100%|██████████| 152/152 [00:00<00:00, 573.14it/s]


In [16]:
data_dict = {
    "train": list_train_data,
    "test": list_test_data
}


for split in ["train", "test"]:
    list_data = data_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_{task}_0219"
    )

Saving the dataset (1/1 shards): 100%|██████████| 152/152 [00:00<00:00, 3765.42 examples/s]
